In [ ]:
import numpy as np
import pandas as pd

from pvlib.tools import cosd
from pvlib import shading
import pvlib
import os

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import satlib

In [ ]:
### Location
lat = 55.6 #degrees North
lon = 12.1 #degrees East
albedo = 0.2

### Dates
start = pd.Timestamp('2023-01-01 00:00:00', tz='Europe/Copenhagen')
end = pd.Timestamp('2023-12-31 23:59:00', tz='Europe/Copenhagen')

### Module parameters
module_width = 1
module_length = 1.6

### System parameters
pitch = 11.75 #estimated from other measurements
length = 3.3 #estimated from other measurements
hub_height = 1.95
gcr = 0.28
max_angle = 60
axis_tilt = 0
axis_azimuth = 180
cross_axis_tilt=0

skymodel = 'perez-driesse' #options: perez, perez-driesse, haydavies, isotropic

In [ ]:
# create location variable
location = pvlib.location.Location(latitude=lat, longitude=lon)
if os.path.isfile(f"data/cams/cams_radiation_{lat}_{lon}_{start}_{end}.csv"):
    data = pd.read_csv(f'data/cams/cams_radiation_{lat}_{lon}_{start}_{end}.csv', index_col=0, parse_dates=True)
else:
    data, meta = pvlib.iotools.get_cams(lat, lon,start,end,'arule@dtu.dk', identifier='cams_radiation',time_step='1min') 
    data.to_csv(f'data/cams/cams_radiation_{lat}_{lon}_{start}_{end}.csv')

In [ ]:
# get solar position and module tilt angle
solar_position = location.get_solarposition(data.index)

#get reference module tilt and POA
module_tilt_ref = pvlib.tracking.singleaxis(solar_position['zenith'], solar_position['azimuth'],backtrack=True, gcr=gcr, max_angle=max_angle,axis_azimuth=180)
surface_orientation_ref = pvlib.tracking.calc_surface_orientation(module_tilt_ref['tracker_theta'], axis_tilt = axis_tilt, axis_azimuth=axis_azimuth)

dni_extra = pvlib.irradiance.get_extra_radiation(solar_position['zenith'])

In [ ]:
omega_ideal = shading.projected_solar_zenith_angle(
    axis_tilt=axis_tilt,
    axis_azimuth=axis_azimuth,
    solar_zenith=solar_position['apparent_zenith'],
    solar_azimuth=solar_position['azimuth'],
)

In [ ]:
#the following is taken from pvlib tracking singleaxis() function

# distance between rows in terms of rack lengths relative to cross-axis tilt
axes_distance = 1/(gcr * cosd(cross_axis_tilt))

#eq. 15 to check for backtracking condition in Anderson, K., and Mikofski, M., "Slope-Aware Backtracking for Single-Axis Trackers", Technical Report NREL/TP-5K00-76626, July 2020. doi:`10.2172/1660126`
temp = np.abs(axes_distance * cosd(omega_ideal - cross_axis_tilt))

# if temp < 1, backtracking is True (see eq. 15 in Anderson and Mikofski, 2020)
backtracking_required = np.where((temp < 1) & np.isfinite(temp), 1, 0)

backtracking_required

In [ ]:
beta_angle_analytical = satlib.diffuse_optimization.munoz2024_analytical(solar_position["zenith"], solar_position["azimuth"], \
                                                   data["dni"], data["dhi"], data["ghi"], albedo)
# beta_angle_analytical = np.clip(beta_angle_analytical, -max_angle, max_angle) #clip to max angle
# beta_angle_analytical = np.where(solar_position["azimuth"] < 180, -1*beta_angle_analytical, beta_angle_analytical)
beta_angle_analytical_2 = satlib.diffuse_optimization.clip_tracking_angle(beta_angle_analytical, module_tilt_ref['tracker_theta'],backtracking_required,max_angle=60)

beta_angle_diffuse = satlib.diffuse_optimization.munoz2024_diffuse(module_tilt_ref['tracker_theta'],solar_position["zenith"], solar_position["azimuth"], \
                                                   dni_extra,data["dni"], data["dhi"], data["ghi"], albedo)

beta_angle_diffuse_2 = satlib.diffuse_optimization.clip_tracking_angle(beta_angle_diffuse, module_tilt_ref['tracker_theta'],backtracking_required,max_angle=60)



In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(beta_angle_analytical),
        mode="lines",
        name="beta_angle_analytical",
        line=dict(color="red"),
    ),
)

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(beta_angle_analytical_2),
        mode="lines",
        name="beta_angle_analytical_2",
        line=dict(color="blue"),
    ),
)

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(beta_angle_diffuse),
        mode="lines",
        name="beta_angle_diffuse",
        line=dict(color="orange"),
    ),
)

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(beta_angle_diffuse_2),
        mode="lines",
        name="beta_angle_diffuse_2",
        line=dict(color="purple"),
    ),
)



fig.add_trace(
    go.Scatter(
        x=data.index,
        y=backtracking_required,
        mode="lines",
        name="backtracking",
        line=dict(color="green"),
    ),
    secondary_y=True,
)

# fig.add_trace(
#     go.Scatter(
#         x=data.index,
#         y=upper_masked,
#         mode="lines",
#         name="Backtracking region",
#         fill="tonexty",
#         fillcolor="rgba(255, 165, 0, 0.25)",
#         line=dict(color="red"),
#     )
# )
